# Vector Space Models and Embeddings in RAGs

<a href="https://colab.research.google.com/github/EasyLearnJava/AgenticProgramming/blob/main/projects/RAG/IT%20Knowledge%20Base/RAG%20series/1.Vector%20Space%20Models%20and%20Embeddings%20in%20RAGs/vector_space_models_and_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code for course 1 of *Retrieval Augmented Generation (RAG) for Developers*. The course explains its ideas with slides and no code, so each cell below shows one of them on real text. It uses a small embedding model, `all-MiniLM-L6-v2`, that runs on an ordinary CPU, and no API key.

Run the cells in order: in Colab, **Runtime → Run all**. The first run downloads the model, about 90 MB.

In [1]:
# In Google Colab: install what it doesn't have yet. Anywhere else, install requirements.txt first.
import sys

if "google.colab" in sys.modules:
    %pip install -q sentence-transformers scikit-learn

## 1. An embedding is a list of numbers

An embedding model turns any text into a vector: here, 384 numbers. Texts with similar meanings get similar numbers. The word *bank* is in all three sentences, but the model reads the words around it.

In [2]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")   # small, runs on a CPU
sentences = ["I deposited money at the bank",
             "The bank approved my loan",
             "We had a picnic on the river bank"]
emb = model.encode(sentences)
print(emb.shape)
print(util.cos_sim(emb, emb).numpy().round(3))

(3, 384)
[[1.    0.486 0.416]
 [0.486 1.    0.288]
 [0.416 0.288 1.   ]]


Each sentence became 384 numbers. The grid compares every pair (1 = the same meaning): the two money sentences are closest, and the river sentence is furthest from the loan sentence.

## 2. TF-IDF counts words, so it can't see meaning

TF-IDF gives each word a score: how often it appears in a document (TF), weighted by how rare it is across all the documents (IDF). Each document becomes a sparse vector, one number per word.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = ["I drive my car to work",
        "The automobile needs new tires",
        "My car needs new tires"]
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(docs)        # sparse: one column per distinct word
print(X.shape)
print(tfidf.get_feature_names_out())
print(cosine_similarity(X).round(3))

(3, 10)
['automobile' 'car' 'drive' 'my' 'needs' 'new' 'the' 'tires' 'to' 'work']
[[1.    0.    0.334]
 [0.    1.    0.528]
 [0.334 0.528 1.   ]]


The first two documents are both about a car, yet their similarity is 0.0: they share no word. Embeddings don't need shared words: section 4 scores *car* and *automobile* 0.865.

## 3. Word2Vec's idea: king − man + woman ≈ queen

Word2Vec learns word vectors from the contexts words appear in, so vector arithmetic follows meaning. The same idea, tried with this notebook's model on a short list of candidate words:

In [4]:
vocab = ["queen", "princess", "prince", "monarch", "woman", "man", "king", "mountain", "river", "girl"]
V = {w: model.encode(w) for w in vocab}
target = V["king"] - V["man"] + V["woman"]
ranked = sorted(((util.cos_sim(target, V[w]).item(), w) for w in vocab
                 if w not in ("king", "man", "woman")), reverse=True)
print([(w, round(s, 3)) for s, w in ranked[:4]])

[('queen', 0.579), ('monarch', 0.547), ('princess', 0.442), ('prince', 0.388)]


## 4. Search, match, rank

The three steps that make embeddings the engine of retrieval. **Semantic search**: a query finds text that means the same thing, in other words.

In [5]:
query = "AI writing tools"
docs = ["Chatbots for content creation",
        "Political debates on energy policy",
        "Global warming impacts on agriculture"]
scores = util.cos_sim(model.encode(query), model.encode(docs))[0]
for doc, score in sorted(zip(docs, scores.tolist()), key=lambda x: x[1], reverse=True):
    print(f"{score:.3f}  {doc}")

0.486  Chatbots for content creation
0.100  Global warming impacts on agriculture
0.035  Political debates on energy policy


**Similarity matching**: how close two vectors are encodes how similar the texts are.

In [6]:
pairs = [("doctor", "physician"), ("doctor", "banana"), ("car", "automobile"), ("car", "banana")]
for a, b in pairs:
    print(f"{a} · {b}: {util.cos_sim(model.encode(a), model.encode(b)).item():.3f}")

doctor · physician: 0.860
doctor · banana: 0.338
car · automobile: 0.865
car · banana: 0.390


**Relevance ranking**: candidates are sorted by similarity, so the most relevant comes first, even when it shares no words with the query.

In [7]:
query = "climate change effects"
docs = ["Political debates on energy policy",
        "Global warming impacts on agriculture"]
scores = util.cos_sim(model.encode(query), model.encode(docs))[0]
for doc, score in sorted(zip(docs, scores.tolist()), key=lambda x: x[1], reverse=True):
    print(f"{score:.3f}  {doc}")

0.641  Global warming impacts on agriculture
0.253  Political debates on energy policy


## 5. Three similarity metrics

**Cosine similarity** measures the angle between two vectors, **Euclidean distance** the straight line between them, and the **dot product** direction and length together. Three short vectors show the difference: `b` points the same way as `a` but is twice as long, and `c` is at a right angle to `a`.

In [8]:
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([2.0, 4.0])      # same direction as a, twice as long
c = np.array([2.0, -1.0])     # at a right angle to a
cos = lambda x, y: x @ y / (np.linalg.norm(x) * np.linalg.norm(y))
print(f"a,b  cosine {cos(a, b):.3f}  euclidean {np.linalg.norm(a - b):.3f}  dot {a @ b:.3f}")
print(f"a,c  cosine {cos(a, c):.3f}  euclidean {np.linalg.norm(a - c):.3f}  dot {a @ c:.3f}")
print("this model's vectors have length", np.linalg.norm(model.encode("anything")).round(4))

a,b  cosine 1.000  euclidean 2.236  dot 10.000
a,c  cosine 0.000  euclidean 3.162  dot 0.000
this model's vectors have length 1.0


Cosine says `a` and `b` are identical; Euclidean distance still sees a gap, because `b` is longer. At a right angle (`a`, `c`), cosine and the dot product are both 0. The model's own vectors all have length 1, and for those, cosine similarity and the dot product are the same number.

## 6. Dimensionality reduction with PCA

PCA keeps the directions along which the vectors differ most. Squeezed from 384 numbers to 2, these words land in groups: people, nature, money.

In [9]:
from sklearn.decomposition import PCA

words = ["king", "queen", "man", "woman", "mountain", "river", "money", "finance"]
xy = PCA(n_components=2).fit_transform(model.encode(words))   # 384 numbers → 2 per word
for w, (x, y) in zip(words, xy):
    print(f"{w:9s} {x:+.2f} {y:+.2f}")

king      -0.42 +0.08
queen     -0.34 +0.39
man       -0.21 +0.08
woman     -0.07 +0.34
mountain  -0.23 -0.65
river     +0.12 -0.44
money     +0.52 +0.06
finance   +0.62 +0.13


## 7. How fast storage grows

Every number is stored as a 4-byte float. A billion vectors of 768 numbers each:

In [10]:
n, d = 1_000_000_000, 768
print(f"{n * d * 4 / 1e12:.2f} TB as float32; {n * 256 * 4 / 1e12:.2f} TB at 256 dimensions")

3.07 TB as float32; 1.02 TB at 256 dimensions


## Next

These embeddings are what a RAG system stores and searches. To see them stored in a vector database and searched with a question, open the [Entry V1 notebook](https://github.com/EasyLearnJava/AgenticProgramming/blob/main/projects/RAG/IT%20Knowledge%20Base/Entry/V1/rag_basics.ipynb). The same steps with Gemini embeddings and the Qdrant database: [Basic 04 · Embeddings](https://github.com/EasyLearnJava/AgenticProgramming/tree/main/projects/RAG/IT%20Knowledge%20Base/Basic/04-embeddings) and [Basic 05 · Vector store](https://github.com/EasyLearnJava/AgenticProgramming/tree/main/projects/RAG/IT%20Knowledge%20Base/Basic/05-vector-store).